# Regime-Adaptive Residual Wavelet Forecaster

This notebook implements one **serious next direction** after your failed PBT, DBLoss, and adaptive-lookback runs:

- **Base expert**: simple target-channel DLinear-style predictor
- **Residual expert**: wavelet forecasting expert using your existing `AdaptiveWaveletMixer`
- **Gate**: learns when to turn the wavelet residual on, based on spectral-complexity signals
- **Pilot first**: compares
  1. `base_only`
  2. `fixed_residual`
  3. `gated_residual` (main idea)

If the pilot is good enough, it can automatically run the full 9-dataset benchmark for the `gated_residual` model.

## Strongest defensible claim
A **regime-adaptive wavelet residual corrector** can reduce the harm caused by always-on wavelet processing in simple regimes, while preserving gains in spectrally complex regimes.

## Weakest claim you should NOT make
Do **not** claim state-of-the-art across all 9 datasets unless the full run really proves it.

## Minimal kill test
If `gated_residual` does **not** beat `fixed_residual` on at least 3/4 pilot datasets, or does not clearly reduce damage on simple regimes, reject the idea.

In [ ]:
# ============================================
# CONFIG
# ============================================

RUN_MODE = "pilot_then_full"   # options: "pilot_only", "pilot_then_full", "full_only"
AUTO_FULL_IF_PILOT_GOOD = True

PILOT_DATASETS = ["etth1", "weather", "traffic", "ili"]

# pilot success rule:
# 1) gated_residual beats fixed_residual on >= 3 pilot datasets by RMSE
# 2) gated_residual beats base_only on >= 2 pilot datasets by RMSE
PILOT_THRESH_FIXED = 3
PILOT_THRESH_BASE = 2

SEED = 42
ROOT_DIR = "/data/Sajjan_Singh/spml/wavelet_seq_project"

In [ ]:
# ============================================
# IMPORTS AND PROJECT SETUP
# ============================================

from pathlib import Path
import sys
import importlib
import gc
import json
import random
import copy
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path(ROOT_DIR).resolve()
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

OUT_TABLES = ROOT / "results" / "tables" / "phase10_regime_adaptive"
OUT_PREDS = ROOT / "results" / "predictions" / "phase10_regime_adaptive"
OUT_CKPTS = ROOT / "results" / "checkpoints" / "phase10_regime_adaptive"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_PREDS.mkdir(parents=True, exist_ok=True)
OUT_CKPTS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = ROOT / "results" / "tables" / "master_all_models_metrics_unified.csv"
BEST_PATH = ROOT / "results" / "tables" / "master_best_by_dataset_unified.csv"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
else:
    GPU_NAME = "CPU"
    GPU_MEM_GB = 0.0

USE_AMP = bool(torch.cuda.is_available()) and bool(getattr(p5, "TRAIN_CFG", {}).get("use_amp", True))
PATIENCE = int(getattr(p5, "TRAIN_CFG", {}).get("full_patience", 8))

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE:", p5.DEVICE)
print("GPU_NAME:", GPU_NAME)
print("GPU_MEM_GB:", round(GPU_MEM_GB, 2))
print("USE_AMP:", USE_AMP)
print("PATIENCE:", PATIENCE)
print("All datasets:", p5.ALL_DATASETS)

In [ ]:
# ============================================
# FIXED ARCHITECTURE (from your best ablation)
# ============================================

ALL_DATASETS = list(p5.ALL_DATASETS)

BASE_ARCH = {
    "levels": 2,
    "wavelet_family": "Db4",
    "d_model": 128,
    "num_backbone_blocks": 2,
    "dropout": 0.10,
    "filter_reg_lambda": 1e-4,
    "gate_entropy_lambda": 1e-4,
}

if GPU_MEM_GB >= 80:
    BATCH_MUL = {
        "etth1": 2, "etth2": 2, "ettm1": 2, "ettm2": 2,
        "weather": 2, "electricity": 1, "traffic": 1,
        "exchange": 2, "ili": 2,
    }
else:
    BATCH_MUL = {ds: 1 for ds in ALL_DATASETS}

REGIME_CFG = {
    "lambda_gate_align": 5e-3,
    "lambda_gate_sparse": 5e-4,
    "lambda_gate_entropy": 5e-4,
    "lambda_residual_aux": 0.20,
    "gate_hidden": 32,
}

print("BASE_ARCH:", BASE_ARCH)
print("REGIME_CFG:", REGIME_CFG)
print("BATCH_MUL:", BATCH_MUL)

In [ ]:
# ============================================
# HELPERS
# ============================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))

def resolve_seq_len(ds):
    seq_candidates = p5.resolve_seq_candidates(ds, "searched")
    return int(seq_candidates[min(1, len(seq_candidates) - 1)])

def resolve_batch_size(ds):
    return int(p5.DEFAULT_BATCH_SIZE[ds] * BATCH_MUL[ds])

def resolve_epochs(ds):
    return int(p5.DEFAULT_EPOCHS[ds])

def moving_avg_1d(x, kernel):
    if kernel <= 1:
        return x
    pad = kernel // 2
    x3 = x.unsqueeze(1)
    xpad = F.pad(x3, (pad, pad), mode="replicate")
    out = F.avg_pool1d(xpad, kernel_size=kernel, stride=1)
    return out.squeeze(1)

def metric_row_from_npz(dataset, model, family, seq_len, pred_len, pred_file_rel):
    p = ROOT / str(pred_file_rel)
    arr = np.load(p, allow_pickle=True)
    preds = np.asarray(arr["preds"], dtype=np.float64).reshape(-1)
    trues = np.asarray(arr["trues"], dtype=np.float64).reshape(-1)
    return {
        "dataset": dataset,
        "model": model,
        "family": family,
        "seq_len": int(seq_len),
        "pred_len": int(pred_len),
        "test_mse": mse(trues, preds),
        "test_mae": mae(trues, preds),
        "test_rmse": rmse(trues, preds),
        "prediction_file": str(pred_file_rel),
    }

In [ ]:
# ============================================
# MODEL COMPONENTS
# ============================================

class DLinearTargetOnly(nn.Module):
    def __init__(self, seq_len, pred_len, kernel_size=25):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.kernel_size = int(kernel_size)
        self.trend_linear = nn.Linear(self.seq_len, self.pred_len)
        self.seasonal_linear = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x_target):
        trend = moving_avg_1d(x_target, self.kernel_size)
        seasonal = x_target - trend
        out = self.trend_linear(trend) + self.seasonal_linear(seasonal)
        return out

def compute_complexity_proxy(x_target):
    B, T = x_target.shape
    spec = torch.fft.rfft(x_target, dim=1)
    power = spec.real.pow(2) + spec.imag.pow(2)
    total_power = power.sum(dim=1) + 1e-8

    Fbins = power.shape[1]
    split = max(1, int(round(Fbins * 0.50)))
    high_power = power[:, split:].sum(dim=1)
    high_ratio = high_power / total_power

    tv = (x_target[:, 1:] - x_target[:, :-1]).abs().mean(dim=1)
    scale = x_target.abs().mean(dim=1) + 1e-6
    tv_norm = torch.tanh(tv / scale)

    x1 = x_target[:, 1:]
    x0 = x_target[:, :-1]
    num = (x1 * x0).mean(dim=1)
    den = x1.pow(2).mean(dim=1).sqrt() * x0.pow(2).mean(dim=1).sqrt() + 1e-6
    lag1_corr = num / den
    decor = 1.0 - lag1_corr.abs()

    proxy = 0.50 * high_ratio + 0.30 * tv_norm + 0.20 * decor
    return proxy.clamp(0.0, 1.0)

class RegimeGate(nn.Module):
    def __init__(self, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x_target):
        proxy = compute_complexity_proxy(x_target)
        mean_abs = x_target.abs().mean(dim=1)
        std = x_target.std(dim=1)
        if x_target.shape[1] >= 2:
            last_diff = (x_target[:, -1] - x_target[:, -2]).abs()
        else:
            last_diff = torch.zeros_like(mean_abs)

        feats = torch.stack([
            proxy,
            torch.tanh(std / (mean_abs + 1e-6)),
            torch.tanh(last_diff / (mean_abs + 1e-6)),
            torch.ones_like(proxy),
        ], dim=1)

        alpha = torch.sigmoid(self.net(feats)).squeeze(-1)
        return alpha, proxy

class ResidualWaveletExpert(nn.Module):
    def __init__(self, input_dim, pred_len):
        super().__init__()
        self.model = p5.AdaptiveWaveletMixer(
            input_dim=int(input_dim),
            pred_len=int(pred_len),
            d_model=int(BASE_ARCH["d_model"]),
            levels=int(BASE_ARCH["levels"]),
            wavelet_family=BASE_ARCH["wavelet_family"],
            num_backbone_blocks=int(BASE_ARCH["num_backbone_blocks"]),
            dropout=float(BASE_ARCH["dropout"]),
            filter_reg_lambda=float(BASE_ARCH["filter_reg_lambda"]),
            gate_entropy_lambda=float(BASE_ARCH["gate_entropy_lambda"]),
        )

    def forward(self, x_scaled):
        pred_scaled, aux = self.model(x_scaled)
        return pred_scaled, aux

class RegimeAdaptiveResidualWavelet(nn.Module):
    def __init__(self, seq_len, pred_len, input_dim, target_idx, mode="gated_residual"):
        super().__init__()
        assert mode in ["base_only", "fixed_residual", "gated_residual"]
        self.mode = mode
        self.target_idx = int(target_idx)
        self.base_expert = DLinearTargetOnly(seq_len=seq_len, pred_len=pred_len, kernel_size=25)
        self.wavelet_expert = ResidualWaveletExpert(input_dim=input_dim, pred_len=pred_len)
        self.gate = RegimeGate(hidden=REGIME_CFG["gate_hidden"])

    def forward(self, x_scaled):
        x_target = x_scaled[:, :, self.target_idx]
        base_pred = self.base_expert(x_target)

        if self.mode == "base_only":
            aux = {
                "alpha": torch.zeros(x_scaled.shape[0], device=x_scaled.device),
                "complexity_proxy": compute_complexity_proxy(x_target),
                "base_pred": base_pred,
                "resid_pred": torch.zeros_like(base_pred),
            }
            return base_pred, aux

        resid_pred, wave_aux = self.wavelet_expert(x_scaled)

        if self.mode == "fixed_residual":
            alpha = torch.ones(x_scaled.shape[0], device=x_scaled.device)
            proxy = compute_complexity_proxy(x_target)
        else:
            alpha, proxy = self.gate(x_target)

        final_pred = base_pred + alpha.unsqueeze(1) * resid_pred
        aux = {
            "alpha": alpha,
            "complexity_proxy": proxy,
            "base_pred": base_pred,
            "resid_pred": resid_pred,
        }
        return final_pred, aux

def hybrid_loss(pred_scaled, y_scaled, aux, mode):
    base_loss = F.mse_loss(pred_scaled, y_scaled)

    if mode == "base_only":
        return base_loss, {
            "final_mse": float(base_loss.detach().cpu().item()),
            "gate_align": 0.0,
            "gate_sparse": 0.0,
            "gate_entropy": 0.0,
            "resid_aux": 0.0,
            "alpha_mean": 0.0,
        }

    alpha = aux["alpha"]
    proxy = aux["complexity_proxy"]
    base_pred = aux["base_pred"]
    resid_pred = aux["resid_pred"]

    resid_target = (y_scaled - base_pred).detach()
    resid_aux = F.mse_loss(resid_pred, resid_target)

    if mode == "fixed_residual":
        total = base_loss + REGIME_CFG["lambda_residual_aux"] * resid_aux
        return total, {
            "final_mse": float(base_loss.detach().cpu().item()),
            "gate_align": 0.0,
            "gate_sparse": 0.0,
            "gate_entropy": 0.0,
            "resid_aux": float(resid_aux.detach().cpu().item()),
            "alpha_mean": float(alpha.mean().detach().cpu().item()),
        }

    gate_align = F.mse_loss(alpha, proxy)
    gate_sparse = alpha.mean()
    gate_entropy = -(alpha.clamp_min(1e-8).log() * alpha + (1 - alpha).clamp_min(1e-8).log() * (1 - alpha)).mean()

    total = (
        base_loss
        + REGIME_CFG["lambda_residual_aux"] * resid_aux
        + REGIME_CFG["lambda_gate_align"] * gate_align
        + REGIME_CFG["lambda_gate_sparse"] * gate_sparse
        + REGIME_CFG["lambda_gate_entropy"] * gate_entropy
    )

    return total, {
        "final_mse": float(base_loss.detach().cpu().item()),
        "gate_align": float(gate_align.detach().cpu().item()),
        "gate_sparse": float(gate_sparse.detach().cpu().item()),
        "gate_entropy": float(gate_entropy.detach().cpu().item()),
        "resid_aux": float(resid_aux.detach().cpu().item()),
        "alpha_mean": float(alpha.mean().detach().cpu().item()),
    }

In [ ]:
# ============================================
# TRAIN / EVAL
# ============================================

@torch.no_grad()
def evaluate_model(model, loader, target_mean, target_std):
    model.eval()
    preds = []
    trues = []
    alpha_rows = []
    proxy_rows = []

    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_scaled, aux = model(x)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)

        alpha_rows.append(aux["alpha"].detach().cpu().numpy())
        proxy_rows.append(aux["complexity_proxy"].detach().cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    alpha = np.concatenate(alpha_rows, axis=0)
    proxy = np.concatenate(proxy_rows, axis=0)

    return {
        "preds": preds,
        "trues": trues,
        "mse": mse(trues.reshape(-1), preds.reshape(-1)),
        "mae": mae(trues.reshape(-1), preds.reshape(-1)),
        "rmse": rmse(trues.reshape(-1), preds.reshape(-1)),
        "avg_alpha": float(alpha.mean()),
        "avg_proxy": float(proxy.mean()),
    }

def train_one_run(dataset_name, mode):
    set_seed(SEED)

    bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
    pred_len = p5.resolve_pred_len(dataset_name, "long")
    seq_len = resolve_seq_len(dataset_name)
    batch_size = resolve_batch_size(dataset_name)
    epochs = resolve_epochs(dataset_name)

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = RegimeAdaptiveResidualWavelet(
        seq_len=seq_len,
        pred_len=pred_len,
        input_dim=int(bundle["values_scaled"].shape[1]),
        target_idx=int(target_idx),
        mode=mode,
    ).to(p5.DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
        foreach=False,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_rmse = float("inf")
    best_epoch = -1
    best_state = None
    patience_left = PATIENCE

    history = []
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        final_mse_parts = []
        gate_align_parts = []
        gate_sparse_parts = []
        gate_entropy_parts = []
        resid_aux_parts = []
        alpha_parts = []

        for batch in train_loader:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred_scaled, aux = model(x)
                loss, parts = hybrid_loss(pred_scaled, y, aux, mode=mode)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            losses.append(float(loss.detach().cpu().item()))
            final_mse_parts.append(parts["final_mse"])
            gate_align_parts.append(parts["gate_align"])
            gate_sparse_parts.append(parts["gate_sparse"])
            gate_entropy_parts.append(parts["gate_entropy"])
            resid_aux_parts.append(parts["resid_aux"])
            alpha_parts.append(parts["alpha_mean"])

        val_eval = evaluate_model(model, val_loader, target_mean, target_std)
        val_rmse = float(val_eval["rmse"])
        mean_train_loss = float(np.mean(losses)) if len(losses) > 0 else np.nan

        history.append({
            "dataset": dataset_name,
            "mode": mode,
            "epoch": epoch,
            "train_loss": mean_train_loss,
            "train_final_mse": float(np.mean(final_mse_parts)),
            "train_gate_align": float(np.mean(gate_align_parts)),
            "train_gate_sparse": float(np.mean(gate_sparse_parts)),
            "train_gate_entropy": float(np.mean(gate_entropy_parts)),
            "train_resid_aux": float(np.mean(resid_aux_parts)),
            "train_alpha_mean": float(np.mean(alpha_parts)),
            "val_rmse": val_rmse,
            "val_mae": float(val_eval["mae"]),
            "val_mse": float(val_eval["mse"]),
            "val_avg_alpha": float(val_eval["avg_alpha"]),
            "val_avg_proxy": float(val_eval["avg_proxy"]),
        })

        print(
            f"[{dataset_name} | {mode}] epoch {epoch:03d} | "
            f"train_loss={mean_train_loss:.6f} | val_rmse={val_rmse:.6f} | "
            f"alpha={val_eval['avg_alpha']:.4f} | proxy={val_eval['avg_proxy']:.4f} | "
            f"batch_size={batch_size}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{dataset_name} | {mode}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    test_eval = evaluate_model(model, test_loader, target_mean, target_std)

    ckpt_path = OUT_CKPTS / f"{dataset_name}_{mode}.pt"
    pred_path = OUT_PREDS / f"{dataset_name}_{mode}_test_predictions.npz"

    torch.save({
        "dataset": dataset_name,
        "mode": mode,
        "base_arch": BASE_ARCH,
        "regime_cfg": REGIME_CFG,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    np.savez_compressed(
        pred_path,
        preds=test_eval["preds"],
        trues=test_eval["trues"],
        seq_len=seq_len,
        pred_len=pred_len,
        avg_alpha=np.array([test_eval["avg_alpha"]], dtype=np.float32),
        avg_proxy=np.array([test_eval["avg_proxy"]], dtype=np.float32),
    )

    row = {
        "dataset": dataset_name,
        "mode": mode,
        "model": f"RegimeAdaptiveResidualWavelet_{mode}",
        "family": "wavelet_regime_adaptive",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "batch_size": batch_size,
        "epochs_target": epochs,
        "best_epoch": int(best_epoch),
        "train_seconds": float(time.time() - t0),
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(test_eval["mse"]),
        "test_mae": float(test_eval["mae"]),
        "test_rmse": float(test_eval["rmse"]),
        "avg_test_alpha": float(test_eval["avg_alpha"]),
        "avg_test_proxy": float(test_eval["avg_proxy"]),
    }

    hist_df = pd.DataFrame(history)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, hist_df

In [ ]:
# ============================================
# PILOT RUN
# ============================================

pilot_rows = []
pilot_hist = []

if RUN_MODE in ["pilot_only", "pilot_then_full"]:
    print("Starting pilot...")
    for ds in PILOT_DATASETS:
        for mode in ["base_only", "fixed_residual", "gated_residual"]:
            print("\n" + "=" * 120)
            print(f"PILOT | dataset={ds} | mode={mode}")
            print("=" * 120)
            row, hist_df = train_one_run(ds, mode)
            pilot_rows.append(row)
            pilot_hist.append(hist_df)

    pilot_df = pd.DataFrame(pilot_rows)
    pilot_hist_df = pd.concat(pilot_hist, ignore_index=True)

    pilot_metrics_csv = OUT_TABLES / "pilot_metrics.csv"
    pilot_history_csv = OUT_TABLES / "pilot_history.csv"
    pilot_df.to_csv(pilot_metrics_csv, index=False)
    pilot_hist_df.to_csv(pilot_history_csv, index=False)

    print("\nSaved:", pilot_metrics_csv)
    print("Saved:", pilot_history_csv)
    display(pilot_df)

    pivot = pilot_df.pivot(index="dataset", columns="mode", values="test_rmse").reset_index()
    pivot["gated_vs_fixed_gain"] = pivot["fixed_residual"] - pivot["gated_residual"]
    pivot["gated_vs_base_gain"] = pivot["base_only"] - pivot["gated_residual"]

    fixed_wins = int((pivot["gated_vs_fixed_gain"] > 0).sum())
    base_wins = int((pivot["gated_vs_base_gain"] > 0).sum())

    pilot_summary = pivot.copy()
    pilot_summary["pilot_pass_fixed"] = fixed_wins
    pilot_summary["pilot_pass_base"] = base_wins

    pilot_summary_csv = OUT_TABLES / "pilot_summary.csv"
    pilot_summary.to_csv(pilot_summary_csv, index=False)

    print("\nSaved:", pilot_summary_csv)
    display(pilot_summary)

    PILOT_GOOD = (fixed_wins >= PILOT_THRESH_FIXED) and (base_wins >= PILOT_THRESH_BASE)
    print("\nPILOT DECISION")
    print(f"gated_residual beats fixed_residual on {fixed_wins}/{len(PILOT_DATASETS)} pilot datasets")
    print(f"gated_residual beats base_only on {base_wins}/{len(PILOT_DATASETS)} pilot datasets")
    print("PILOT_GOOD =", PILOT_GOOD)
else:
    PILOT_GOOD = False

In [ ]:
# ============================================
# FULL RUN (gated_residual only) IF PILOT IS GOOD
# ============================================

full_rows = []
full_hist = []

RUN_FULL = (
    (RUN_MODE == "full_only") or
    (RUN_MODE == "pilot_then_full" and AUTO_FULL_IF_PILOT_GOOD and PILOT_GOOD)
)

print("RUN_FULL =", RUN_FULL)

if RUN_FULL:
    for ds in ALL_DATASETS:
        print("\n" + "=" * 120)
        print(f"FULL RUN | dataset={ds} | mode=gated_residual")
        print("=" * 120)
        row, hist_df = train_one_run(ds, "gated_residual")
        full_rows.append(row)
        full_hist.append(hist_df)

    full_df = pd.DataFrame(full_rows)
    full_hist_df = pd.concat(full_hist, ignore_index=True)

    full_metrics_csv = OUT_TABLES / "full_metrics.csv"
    full_history_csv = OUT_TABLES / "full_history.csv"
    full_df.to_csv(full_metrics_csv, index=False)
    full_hist_df.to_csv(full_history_csv, index=False)

    print("\nSaved:", full_metrics_csv)
    print("Saved:", full_history_csv)
    display(full_df)
else:
    print("Full run skipped. Stop here if pilot failed.")

In [ ]:
# ============================================
# MERGE WITH CURRENT MASTER AND COMPARE AGAINST CURRENT BEST
# ============================================

if RUN_FULL:
    if not MASTER_PATH.exists():
        raise FileNotFoundError(f"Missing master file: {MASTER_PATH}")
    if not BEST_PATH.exists():
        raise FileNotFoundError(f"Missing best file: {BEST_PATH}")

    master = pd.read_csv(MASTER_PATH)
    best_old = pd.read_csv(BEST_PATH)

    master_candidate = master[master["model"] != "RegimeAdaptiveResidualWavelet_gated_residual"].copy()

    new_rows = []
    for _, r in full_df.iterrows():
        new_rows.append(
            metric_row_from_npz(
                dataset=r["dataset"],
                model=r["model"],
                family=r["family"],
                seq_len=r["seq_len"],
                pred_len=r["pred_len"],
                pred_file_rel=r["prediction_file"],
            )
        )

    new_df = pd.DataFrame(new_rows)
    master_candidate = pd.concat([master_candidate, new_df], ignore_index=True)
    master_candidate = master_candidate.sort_values(["dataset", "family", "model"]).reset_index(drop=True)

    best_candidate = (
        master_candidate.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
                        .groupby("dataset", as_index=False)
                        .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
    )

    candidate_master_csv = ROOT / "results" / "tables" / "master_all_models_metrics_unified_regime_candidate.csv"
    candidate_best_csv = ROOT / "results" / "tables" / "master_best_by_dataset_unified_regime_candidate.csv"

    master_candidate.to_csv(candidate_master_csv, index=False)
    best_candidate.to_csv(candidate_best_csv, index=False)

    compare = best_old.rename(columns={
        "model": "old_best_model",
        "family": "old_best_family",
        "test_mse": "old_best_mse",
        "test_mae": "old_best_mae",
        "test_rmse": "old_best_rmse",
    }).merge(
        full_df[["dataset", "model", "test_mse", "test_mae", "test_rmse", "avg_test_alpha", "avg_test_proxy"]].rename(columns={
            "model": "regime_model",
            "test_mse": "regime_mse",
            "test_mae": "regime_mae",
            "test_rmse": "regime_rmse",
        }),
        on="dataset",
        how="left"
    )

    compare["rmse_gain_abs"] = compare["old_best_rmse"] - compare["regime_rmse"]
    compare["rmse_gain_pct"] = 100.0 * compare["rmse_gain_abs"] / compare["old_best_rmse"]

    compare["mae_gain_abs"] = compare["old_best_mae"] - compare["regime_mae"]
    compare["mae_gain_pct"] = 100.0 * compare["mae_gain_abs"] / compare["old_best_mae"]

    compare_csv = OUT_TABLES / "full_vs_current_best.csv"
    compare.to_csv(compare_csv, index=False)

    wins = int((compare["rmse_gain_abs"] > 0).sum())
    losses = int((compare["rmse_gain_abs"] < 0).sum())
    ties = int((compare["rmse_gain_abs"] == 0).sum())

    print("\nSaved:", candidate_master_csv)
    print("Saved:", candidate_best_csv)
    print("Saved:", compare_csv)
    display(compare)
    display(best_candidate)

    print("\nFINAL DECISION")
    print(f"Regime-adaptive model wins on {wins}/{len(ALL_DATASETS)} datasets by RMSE")
    print(f"Loses on {losses}/{len(ALL_DATASETS)} datasets")
    print(f"Ties on {ties}/{len(ALL_DATASETS)} datasets")

    if wins >= 4:
        print("RECOMMENDATION: promising enough to keep and analyze further.")
    else:
        print("RECOMMENDATION: not strong enough overall; reject or redesign.")
else:
    print("No full-run merge because RUN_FULL is False.")

## What success looks like

For this idea to survive:
- `gated_residual` should beat `fixed_residual` on most pilot datasets
- average gate `alpha` should **not** collapse to ~1 on easy datasets
- easy/simple datasets should show smaller or near-zero residual activation
- harder/spectrally complex datasets should show larger residual activation

## Main output files

Pilot:
- `results/tables/phase10_regime_adaptive/pilot_metrics.csv`
- `results/tables/phase10_regime_adaptive/pilot_summary.csv`

Full:
- `results/tables/phase10_regime_adaptive/full_metrics.csv`
- `results/tables/phase10_regime_adaptive/full_vs_current_best.csv`
- `results/tables/master_best_by_dataset_unified_regime_candidate.csv`